# Fase 5c — Cálculo de embeddings (3 modelos) para los chunks

Para cada uno de los cuatro JSON de chunks generados en la Fase 3b, calcula y añade **tres embeddings distintos** por chunk, con el fin de comparar más adelante qué modelo funciona mejor en la recuperación (Elasticsearch, Fases 5a/5d):

- **`vector_generalista`**: `intfloat/multilingual-e5-base` (modelo E5 multilingüe genérico, con el prefijo `passage:` que exige).
- **`vector_medico`**: `cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR` (modelo biomédico de *entity linking* sobre UMLS, usando *pooling* por token CLS, tal y como fue entrenado, y sin el prefijo E5).
- **`vector_experto`**: el modelo E5 propio, afinado específicamente para este corpus clínico (`./modelo_especializado_e5`, resultado de la Fase 5b).

Solo se vectorizan los chunks que son unidades de recuperación reales: en las estrategias A, B y C, todos los chunks; en la estrategia jerárquica (D), únicamente los chunks **"hijo"** (los "padre" solo aportan contexto adicional al LLM y no se indexan como vectores de búsqueda). El resultado se guarda en ficheros `*_embeddings.json`, listos para insertarse en Elasticsearch (Fase 5d).

In [1]:
!pip install sentence-transformers

In [2]:
import json
import torch
import os
from sentence_transformers import SentenceTransformer, models

# 1. Comprobamos la GPU 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device.upper()}")

# 2. Cargamos los modelos en memoria
print("\nCargando modelo Generalista...")
modelo_general = SentenceTransformer("intfloat/multilingual-e5-base", device=device)

print("\nCargando modelo Médico...")
# Usamos el token CLS porque fue entrenado específicamente así en UMLS
word_embedding_model = models.Transformer('cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR')
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_cls_token=True,
    pooling_mode_mean_tokens=False
)
modelo_medico = SentenceTransformer(modules=[word_embedding_model, pooling_model], device=device)

print("\nCargando modelo E5 Especializado...")
modelo_experto = SentenceTransformer("./modelo_especializado_e5", device=device)

# 3. Función procesadora
def enriquecer_json_con_vectores(ruta_entrada, ruta_salida):
    """
    Calcula y añade a cada chunk del JSON de entrada sus tres embeddings
    (generalista, médico y experto), excluyendo los chunks 'padre' de la
    estrategia jerárquica (solo se vectorizan las unidades realmente
    recuperables: A, B, C y los 'hijo' de D), y guarda el resultado en un
    nuevo JSON con los vectores incorporados.
    """
    print(f"\n==================================================")
    print(f"Procesando archivo: {ruta_entrada}")
    print(f"==================================================")
    
    with open(ruta_entrada, 'r', encoding='utf-8') as f:
        chunks = json.load(f)
        
    print(f"Total de chunks en el archivo: {len(chunks)}")

    # Si no tiene la clave 'doc_type', es de la estrategia A, B o C, así que lo vectorizamos.
    # Si tiene la clave 'doc_type' y es 'hijo', también lo vectorizamos.
    # Si es 'padre', lo ignoramos.
    chunks_a_vectorizar = [
        c for c in chunks 
        if c.get("metadata", {}).get("doc_type", "no_aplica") != "padre"
    ]
    
    print(f"Chunks que realmente necesitan embeddings: {len(chunks_a_vectorizar)}")
    
    # Extraemos solo los textos de los que pasaron el filtro
    textos_crudos = [chunk["text"] for chunk in chunks_a_vectorizar] # Para SapBERT
    textos_e5 = [f"passage: {t}" for t in textos_crudos]             # Para los E5
    
    # Calculamos vectores aprovechando la GPU
    print("Calculando embeddings generalistas (768 dims)...")
    vectores_generales = modelo_general.encode(textos_e5, batch_size=32, show_progress_bar=True)
    
    print("Calculando embeddings médicos (768 dims)...")
    vectores_medicos = modelo_medico.encode(textos_crudos, batch_size=32, show_progress_bar=True)
    
    print("Calculando embeddings e5 expertos (768 dims)...")
    vectores_experto = modelo_experto.encode(textos_e5, batch_size=32, show_progress_bar=True)

    # Añadimos los vectores al diccionario
    for i, chunk in enumerate(chunks_a_vectorizar):
        chunk["vector_generalista"] = vectores_generales[i].tolist()
        chunk["vector_medico"] = vectores_medicos[i].tolist()
        chunk["vector_experto"] = vectores_experto[i].tolist()
        
    # Guardamos el JSON CON OTRO NOMBRE
    with open(ruta_salida, 'w', encoding='utf-8') as f:
        json.dump(chunks, f, ensure_ascii=False, indent=4)
        
    print(f"Archivo guardado con éxito en: {ruta_salida}\n")

# ==========================================
# 4. BUCLE DE PROCESAMIENTO
# ==========================================

archivos_originales = [
    "chunks_estrategia_A_fixed.json",
    "chunks_estrategia_B_markdown.json",
    "chunks_estrategia_C_semantica.json",
    "chunks_estrategia_D_jerarquico.json"
]

# Procesamos los 4 jsons
for archivo_original in archivos_originales:
    # Comprobamos que el archivo existe antes de hacer nada
    if os.path.exists(archivo_original):
        # Generamos el nuevo nombre automáticamente (ej: chunks_estrategia_A_fixed_embeddings.json)
        archivo_nuevo = archivo_original.replace(".json", "_embeddings.json")
        
        # Llamamos a la función
        enriquecer_json_con_vectores(archivo_original, archivo_nuevo)
    else:
        print(f"ERROR: No se encuentra el archivo '{archivo_original}'")

print("¡TODOS LOS ARCHIVOS PROCESADOS Y GUARDADOS!")

Usando dispositivo: CUDA

Cargando modelo Generalista...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Cargando modelo Médico...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Calculando embeddings médicos (768 dims)...


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Calculando embeddings e5 expertos (768 dims)...


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Archivo guardado con éxito en: chunks_estrategia_D_jerarquico_embeddings.json

¡TODOS LOS ARCHIVOS PROCESADOS Y GUARDADOS!
